# Week 7: Deep Learning Foundations
## Notebook 11: Weight Initialization: Xavier Glorot & He Kaiming Derivation

### 1. Conceptual Intuition & Theoretical Background
In deep neural networks ($L > 10$ layers), naive weight initialization leads to catastrophic failure:
- **Weights too large:** Activations explode to $\pm \infty$ or saturate activations (e.g. $\tanh \to \pm 1$), causing gradient vanishing.
- **Weights too small:** Activations exponentially decay to zero in forward pass; gradients vanish to zero during backpropagation.

Consider linear layer $y = \sum_{i=1}^{n_{\text{in}}} w_i x_i$. Assuming zero-mean independent variables:
$$\text{Var}(y) = n_{\text{in}} \text{Var}(w) \text{Var}(x)$$
To keep variance constant ($\text{Var}(y) = \text{Var}(x)$), we must set:
$$\text{Var}(w) = \frac{1}{n_{\text{in}}}$$

**Xavier (Glorot) Initialization (for Tanh / Sigmoid):**
$$\text{Var}(w) = \frac{2}{n_{\text{in}} + n_{\text{out}}} \implies W \sim \mathcal{N}\left(0, \frac{2}{n_{\text{in}} + n_{\text{out}}}\right)$$

**He (Kaiming) Initialization (for ReLU):**
Because ReLU zeros out negative inputs, it cuts activation variance in half. To compensate:
$$\text{Var}(w) = \frac{2}{n_{\text{in}}} \implies W \sim \mathcal{N}\left(0, \frac{2}{n_{\text{in}}}\right)$$

---

### 2. Scratch Python Implementation: 15-Layer Network Variance Tracking

In [ ]:
import numpy as np

# Simulation: Propagate forward through 15 dense layers
n_layers = 15
dim = 256
batch_size = 500

X_init = np.random.randn(batch_size, dim)

def simulate_deep_network(init_type='xavier', activation='tanh'):
    a = X_init.copy()
    layer_std_history = [np.std(a)]
    
    for _ in range(n_layers):
        if init_type == 'naive_small':
            W = np.random.normal(0, 0.01, (dim, dim))
        elif init_type == 'naive_large':
            W = np.random.normal(0, 1.0, (dim, dim))
        elif init_type == 'xavier':
            # Glorot standard deviation
            std = np.sqrt(2.0 / (dim + dim))
            W = np.random.normal(0, std, (dim, dim))
        elif init_type == 'he':
            # Kaiming standard deviation for ReLU
            std = np.sqrt(2.0 / dim)
            W = np.random.normal(0, std, (dim, dim))
            
        z = np.dot(a, W)
        
        if activation == 'tanh':
            a = np.tanh(z)
        elif activation == 'relu':
            a = np.maximum(0, z)
            
        layer_std_history.append(np.std(a))
        
    return layer_std_history

# Compare Initializations with ReLU Activation
std_naive = simulate_deep_network('naive_small', activation='relu')
std_xavier = simulate_deep_network('xavier', activation='relu')
std_he = simulate_deep_network('he', activation='relu')

print("--- Standard Deviation across 15 Layers (ReLU Activation) ---")
print(f"Layer 0 (Input):                {std_he[0]:.4f}")
print(f"Naive Small (0.01) at Layer 15: {std_naive[-1]:.6e} (Collapsed to 0)")
print(f"Xavier at Layer 15:             {std_xavier[-1]:.6e} (Vanished variance)")
print(f"He/Kaiming at Layer 15:         {std_he[-1]:.4f} (Perfect variance preservation!)")

### 3. Step-by-Step Code Walkthrough
1. **Variance Decay in ReLU**: Xavier initialization shrinks variance by a factor of 2 at each layer when using ReLU, collapsing output variance.
2. **He/Kaiming Correction**: The factor $\frac{2}{n_{\text{in}}}$ counterbalances ReLU's half-space clipping.
3. **Stability**: Activations remain well-scaled across arbitrarily deep architectures.

---

### 4. Industry Applications & Trade-offs
- **ResNet & Vision Architectures:** Deep residual networks ($>50$ layers) could not be trained without Kaiming initialization prior to batch normalization.
- **Trade-off:** Modern Transformer architectures often use scaled normal distributions (e.g. $\mathcal{N}(0, 0.02)$) coupled with LayerNorm and Pre-LN.

---

### 5. Top Interview Questions & Answers
**Q1: Why does Xavier initialization cause variance vanishing when applied to ReLU networks?**
> **Answer:** Xavier assumes linear or zero-symmetric activations. ReLU zeroes out roughly half the activations ($x < 0$), halving the propagated variance at every layer: $\text{Var}(y) = \frac{1}{2} n_{\text{in}} \text{Var}(w) \text{Var}(x)$.

**Q2: What is the difference between Kaiming Fan-In vs Fan-Out initialization?**
> **Answer:** Fan-In preserves the variance of input activations in the forward pass ($\text{Var}(w) = 2 / n_{\text{in}}$). Fan-Out preserves the variance of backpropagated gradients in the backward pass ($\text{Var}(w) = 2 / n_{\text{out}}$).